# Date Dimension Notebook: Bronze to Silver

この Notebook は、`bronze.Date` を読み込み、Power BI のタイムインテリジェンスで使いやすい `silver.Date` を作成します。

## この Notebook で行うこと

1. `bronze.Date` のスキーマとデータ品質を確認する
2. `silver.Date` を作成する
   - `Date` を date 型に変換
   - 数値列を integer 型に変換
   - 重複日付を除去
   - 品質フラグと処理時刻を付与

## 想定テーブル
- 入力: `bronze.Date`
- 出力: `silver.Date`

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.sql.window import Window

# 必要に応じて変更してください
LAKEHOUSE_NAME = "demo_lakehouse"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
SOURCE_TABLE = "Date"

BRONZE_DATE_FQN = f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.{SOURCE_TABLE}"
SILVER_DATE_FQN = f"{LAKEHOUSE_NAME}.{SILVER_SCHEMA}.{SOURCE_TABLE}"

SILVER_DATE_PATH = f"Tables/{SILVER_SCHEMA}/{SOURCE_TABLE}"

print("BRONZE_DATE_FQN:", BRONZE_DATE_FQN)
print("SILVER_DATE_FQN:", SILVER_DATE_FQN)

## Bronze Date の確認
元テーブルのスキーマ、件数、重複日付を確認します。

In [ ]:
date_bronze = spark.sql(f"SELECT * FROM {BRONZE_DATE_FQN}")
date_bronze.printSchema()
display(date_bronze.limit(20))

bronze_summary = date_bronze.agg(
    F.count('*').alias('RowCount'),
    F.countDistinct('Date').alias('DistinctDateCount'),
    F.min('Date').alias('MinDateRaw'),
    F.max('Date').alias('MaxDateRaw')
)
display(bronze_summary)

duplicate_dates = (
    date_bronze
    .groupBy('Date')
    .count()
    .filter(F.col('count') > 1)
    .orderBy(F.col('Date'))
)
display(duplicate_dates)

## Silver Date の作成
Silver では、日付ディメンションとして破綻しない状態に整えます。

主な処理:
- `Date` を date 型に変換
- 数値列を integer 型に変換
- 重複日付を除去
- 品質フラグ `DataQualityFlag` を付与
- 処理時刻 `ProcessedTimestamp` を付与

In [ ]:
date_bronze_typed = (
    date_bronze
    .withColumn('Date', F.to_date('Date'))
    .withColumn('Day_Number', F.col('Day_Number').cast(IntegerType()))
    .withColumn('Day', F.col('Day').cast(IntegerType()))
    .withColumn('Calendar_Month_Number', F.col('Calendar_Month_Number').cast(IntegerType()))
    .withColumn('Calendar_Year', F.col('Calendar_Year').cast(IntegerType()))
    .withColumn('Fiscal_Month_Number', F.col('Fiscal_Month_Number').cast(IntegerType()))
    .withColumn('Fiscal_Year', F.col('Fiscal_Year').cast(IntegerType()))
    .withColumn('ISO_Week_Number', F.col('ISO_Week_Number').cast(IntegerType()))
)

silver_quality = (
    date_bronze_typed
    .withColumn(
        'DataQualityFlag',
        F.when(F.col('Date').isNull(), F.lit('Missing Date'))
         .when(F.col('Calendar_Year').isNull(), F.lit('Missing Calendar Year'))
         .when(F.col('Fiscal_Year').isNull(), F.lit('Missing Fiscal Year'))
         .when(F.col('Calendar_Month_Number').isNull(), F.lit('Missing Calendar Month'))
         .when(F.col('Fiscal_Month_Number').isNull(), F.lit('Missing Fiscal Month'))
         .otherwise(F.lit('Valid'))
    )
    .withColumn('ProcessedTimestamp', F.current_timestamp())
)

dedupe_window = Window.partitionBy('Date').orderBy(F.col('ProcessedTimestamp').desc())
silver_ranked = silver_quality.withColumn('RowNumberWithinDate', F.row_number().over(dedupe_window))

date_silver = (
    silver_ranked
    .filter((F.col('DataQualityFlag') == 'Valid') & (F.col('RowNumberWithinDate') == 1))
    .drop('RowNumberWithinDate')
)

display(date_silver.orderBy('Date').limit(20))

silver_summary = date_silver.agg(
    F.count('*').alias('RowCount'),
    F.countDistinct('Date').alias('DistinctDateCount'),
    F.min('Date').alias('MinDate'),
    F.max('Date').alias('MaxDate')
)
display(silver_summary)

## Silver スキーマへの保存
`silver.Date` として保存し、読み戻して確認します。

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
date_silver.write.mode('overwrite').save(SILVER_DATE_PATH)

date_silver_saved = spark.sql(f"SELECT * FROM {SILVER_DATE_FQN} ORDER BY Date")
display(date_silver_saved.limit(20))

## Gold スキーマへの保存
`gold.DimDate` として保存し、Power BI で使う前提の確認を行います。

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
date_gold.write.mode('overwrite').save(GOLD_DATE_PATH)

gold_saved = spark.sql(f"SELECT * FROM {GOLD_DATE_FQN} ORDER BY Date")
display(gold_saved.limit(20))

gold_summary = gold_saved.agg(
    F.count('*').alias('RowCount'),
    F.countDistinct('Date').alias('DistinctDateCount'),
    F.min('Date').alias('MinDate'),
    F.max('Date').alias('MaxDate')
)
display(gold_summary)